# Self-JEV `racing_engineer_v1` — Fresh Racing Qwen3.5-0.8B LoRA (Colab T4)

Trains and calibrates a fresh **Racing Qwen3.5-0.8B LoRA** from `Qwen/Qwen3.5-0.8B-Base` strictly on `RaceEnv` rollout supervision (`data/racing_v1/`):
- **Objective:** Set-valued acceptable-action loss (`completion_then_time_v1`) with balanced build vs. pace loss weighting.
- **Splits (race-lineage isolated):** `train.jsonl` (`853` records from `180` parent races, seeds `3201..3220`) · `dev.jsonl` (`428` records, seeds `3301..3310`) · `cal.jsonl` (`433` records, seeds `3401..3410`).
- **Untouched evaluation panels:** Historical `2001..2005` (`45` cases) and Fresh Confirmation `3501..3520` (`180` cases) are never loaded here.

### Setup
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Upload `racing_v1_colab_bundle.zip` to Google Drive root (`MyDrive/`) or upload when prompted in Cell 2.
3. Run Cells 1–5. All checkpoints (`lora_adapters.pt`, `adapter_config.json`, `calibration.json`) and `training_summary.json` save automatically to `/content/drive/MyDrive/self_jev_racing_v1/`.

In [ ]:
# Cell 1: GPU check, Google Drive mount, and dependencies
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
!pip install -q --upgrade transformers accelerate safetensors pyyaml

In [ ]:
# Cell 2: Locate and unpack racing_v1_colab_bundle.zip
import os, zipfile
from pathlib import Path

os.environ['HF_HUB_DISABLE_XET'] = '1'
WORK_DIR = Path('/content/racing_v1_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

candidates = [
    Path('/content/racing_v1_colab_bundle.zip'),
    Path('/content/drive/MyDrive/racing_v1_colab_bundle.zip'),
]
bundle = next((p for p in candidates if p.is_file()), None)
if bundle is None:
    from google.colab import files
    print('Upload racing_v1_colab_bundle.zip:')
    uploaded = files.upload()
    bundle = Path('/content') / next(iter(uploaded.keys()))

with zipfile.ZipFile(bundle, 'r') as zf:
    zf.extractall(WORK_DIR)

print('Unpacked bundle from:', bundle)
!ls -lh /content/racing_v1_work/data/racing_v1/

In [ ]:
# Cell 3: Fast 16-item smoke test (verifies Qwen3.5-0.8B-Base download, letter IDs, and T4 VRAM)
!python3 /content/racing_v1_work/scripts/train_racing_v1_students.py \
  --data-dir /content/racing_v1_work/data/racing_v1 \
  --out-models-dir /content/racing_v1_work/smoke_models \
  --out-reports-dir /content/racing_v1_work/smoke_reports \
  --student qwen \
  --smoke-test

In [ ]:
# Cell 4: Full 3-seed training + dev selection + cal.jsonl temperature scaling
DRIVE_OUT = '/content/drive/MyDrive/self_jev_racing_v1'
!python3 /content/racing_v1_work/scripts/train_racing_v1_students.py \
  --data-dir /content/racing_v1_work/data/racing_v1 \
  --out-models-dir {DRIVE_OUT}/models \
  --out-reports-dir {DRIVE_OUT}/reports \
  --student qwen \
  --qwen-seeds 42 1024 2026 \
  --qwen-epochs 4 \
  --qwen-lora-r 16 \
  --qwen-lr 2e-4 \
  --grad-accum 8

In [ ]:
# Cell 5: Print 3-seed summary and zip selected checkpoint for local Mac closed-loop eval
import json, zipfile
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/self_jev_racing_v1')
rep = json.loads((DRIVE_OUT / 'reports' / 'training_summary.json').read_text())
q = rep['qwen_lora']

print(f"Selected deployment seed: {q['selected_seed']} (base={q['base_model']}, r={q['lora_r']}, wall={q['wall_s']}s)")
for s in q['seeds']:
    dev = s['dev_calibrated']
    cal = s['cal_calibrated']
    b_acc = dev['by_type']['build']['acceptable_acc']
    p_acc = dev['by_type']['pace']['acceptable_acc']
    print(
        f"  seed={s['seed']:4d} | ep={s['best_epoch']} | T={s['temperature']:.2f} | "
        f"dev_acc={dev['acceptable_acc']:.4f} (build={b_acc:.3f}, pace={p_acc:.3f}) | "
        f"cal_acc={cal['acceptable_acc']:.4f} | ece={dev['ece']:.4f} | vram={s['peak_vram_gb']}GB"
    )

sel_dir = Path(q['selected_checkpoint_dir'])
zip_path = DRIVE_OUT / 'racing_qwen3.5_0.8b_lora_selected.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ('lora_adapters.pt', 'adapter_config.json', 'calibration.json'):
        zf.write(sel_dir / f, f)
print(f"\nSaved selected checkpoint zip -> {zip_path}")
print("On your Mac, unzip into: models/racing_engineer_v1/qwen_lora/racing_qwen3.5_0.8b_lora_s42/")
print("Then run: /opt/homebrew/bin/python3.14 scripts/eval_racing_v1_closed_loop.py")